In [ ]:
import importlib.util
import subprocess
import sys

PACKAGE_SPECS = ["timm>=1.0.0", "openpyxl", "scikit-learn", "matplotlib", "pandas", "numpy", "pillow", "tqdm"]

def import_name_for(package_spec):
    package = package_spec.split(">=")[0].split("==")[0].split("<")[0]
    return {"scikit-learn": "sklearn", "openpyxl": "openpyxl", "pillow": "PIL"}.get(package, package.replace("-", "_"))

missing = [package for package in PACKAGE_SPECS if importlib.util.find_spec(import_name_for(package)) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
print("dependencies_ready")


In [ ]:
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from IPython.display import display
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 260)

SEED = 42
REPEATS = 1
REPEAT_SEEDS = [42]
REPEAT_MODE = "single_fixed_seed"
CLASS_DIRS = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
CLASS_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
NUM_CLASSES = 4
IMG_SIZE = 224
EPOCHS = 30
PATIENCE = 15
PAPER_BATCH_SIZE = 128
MICRO_BATCH_SIZE = int(os.environ.get("CONVNEXT_MICRO_BATCH_SIZE", "32"))
ACCUMULATION_STEPS = max(1, PAPER_BATCH_SIZE // MICRO_BATCH_SIZE)
EVAL_BATCH_SIZE = PAPER_BATCH_SIZE
WARMUP_EPOCHS = 5
WARMUP_HEAD_LR = 1e-3
BACKBONE_FINETUNE_LR = 2e-5
HEAD_FINETUNE_LR = 1e-4
STEP_SIZE = 3
STEP_GAMMA = 0.9
NUM_WORKERS = int(os.environ.get("CONVNEXT_WORKERS", str(min(12, max(4, (os.cpu_count() or 8) // 2)))))
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0
RUN_TRAINING = os.environ.get("RUN_TRAINING", "1").strip() != "0"
PROJECT_ROOT = Path.cwd()
DEFAULT_DATA_DIR = Path("/home/drnguyenvinh/notebooks/uynnhy/processed-images/processed_images")
DATA_DIR_ENV_KEYS = ["SHRIMP_DATA_DIR", "DATA_DIR_OVERRIDE"]
OUTPUT_DIR = Path(os.environ.get("SHRIMP_CONVNEXT_OUTPUT_DIR", str(PROJECT_ROOT / "final_convnext_tiny_shrimpxnet_14losses_repeat1_audited_outputs"))).expanduser().resolve()
REPORT_DIR = OUTPUT_DIR / "reports"
FIGURES_DIR = OUTPUT_DIR / "figures"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
FIXED_SPLIT_MANIFEST_PATH = OUTPUT_DIR / "fixed_split_manifest_seed42_with_md5.csv"
RUN_AUDIT_PATH = OUTPUT_DIR / "run_audit.json"
ENVIRONMENT_VERSIONS_PATH = OUTPUT_DIR / "environment_versions.json"
LOSS_RUN_SUMMARY_RAW_PATH = OUTPUT_DIR / "loss_run_summary_raw.csv"
LOSS_GROUP_STATS_PATH = OUTPUT_DIR / "loss_group_stats.csv"
LOSS_DELTAS_VS_CE_PATH = OUTPUT_DIR / "loss_deltas_vs_ce.csv"
BG_WSSV_ERROR_SUMMARY_PATH = OUTPUT_DIR / "bg_wssv_error_summary.csv"
ALL_PREDICTIONS_PATH = OUTPUT_DIR / "all_predictions.csv"
FINAL_SUMMARY_XLSX_PATH = OUTPUT_DIR / "final_summary.xlsx"
FINAL_SUMMARY_JSON_PATH = OUTPUT_DIR / "final_summary.json"
OUTPUT_TABLE_AUDIT_PATH = OUTPUT_DIR / "output_table_audit.csv"
FIGURES_AND_REPORTS_ZIP_PATH = OUTPUT_DIR / "figures_and_reports.zip"
for directory in [OUTPUT_DIR, REPORT_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def reset_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

reset_all_seeds(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def sanitize_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if math.isnan(float(obj)) else float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return str(obj)

def save_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=json_default)

def nvidia_smi_text():
    try:
        completed = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=8)
        if completed.returncode == 0:
            return completed.stdout
        return completed.stderr
    except Exception as exc:
        return f"{type(exc).__name__}: {exc}"

def has_class_folders(path):
    return all((path / class_dir).exists() for class_dir in CLASS_DIRS)

def data_dir_candidates():
    candidates = []
    for key in DATA_DIR_ENV_KEYS:
        value = os.environ.get(key, "").strip()
        if value:
            candidates.append(Path(value).expanduser())
    candidates.extend([DEFAULT_DATA_DIR, PROJECT_ROOT / "uynnhy" / "processed-images" / "processed_images", PROJECT_ROOT / "uynnhy" / "processed-images", PROJECT_ROOT / "processed_images", PROJECT_ROOT / "processed-images" / "processed_images"])
    unique = []
    seen = set()
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if str(resolved) not in seen:
            seen.add(str(resolved))
            unique.append(resolved)
    return unique

def resolve_data_dir():
    checked = []
    for candidate in data_dir_candidates():
        checked.append(str(candidate))
        if has_class_folders(candidate):
            return candidate
        for child in [candidate / "processed_images", candidate / "processed-images", candidate / "data", candidate / "dataset"]:
            checked.append(str(child))
            if has_class_folders(child):
                return child
    raise FileNotFoundError(json.dumps({"checked_data_dir_candidates": checked}, ensure_ascii=False, indent=2))

DATA_DIR = resolve_data_dir()
print(f"DATA_DIR={DATA_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"device={device}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")


In [ ]:
def discover_processed_images(data_dir):
    if not data_dir.exists():
        raise FileNotFoundError(str(data_dir))
    if not has_class_folders(data_dir):
        missing = [str(data_dir / class_dir) for class_dir in CLASS_DIRS if not (data_dir / class_dir).exists()]
        raise FileNotFoundError(json.dumps(missing, ensure_ascii=False))
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                rows.append({"path": str(path.resolve()), "rel_path": str(path.relative_to(data_dir)), "class_dir": class_dir, "class_name": CLASS_NAMES[CLASS_TO_IDX[class_dir]], "label": CLASS_TO_IDX[class_dir]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"no_images_under_{data_dir}")
    return frame.sort_values("rel_path").reset_index(drop=True)

def add_md5(frame):
    out = frame.copy().reset_index(drop=True)
    missing = [path for path in out["path"].tolist() if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(missing[0])
    out["md5"] = [md5_file(Path(path)) for path in out["path"].tolist()]
    return out

def validate_source_split(train_frame, val_frame, test_frame):
    for split_name, frame in [("train", train_frame), ("val", val_frame), ("test", test_frame)]:
        if not frame["rel_path"].is_unique:
            raise RuntimeError(f"duplicate_rel_path_{split_name}")
        missing = [path for path in frame["path"].tolist() if not Path(path).exists()]
        if missing:
            raise FileNotFoundError(missing[0])
    sets = {"train": set(train_frame["rel_path"]), "val": set(val_frame["rel_path"]), "test": set(test_frame["rel_path"])}
    overlaps = {"train_val": sorted(sets["train"] & sets["val"]), "train_test": sorted(sets["train"] & sets["test"]), "val_test": sorted(sets["val"] & sets["test"])}
    if any(overlaps.values()):
        raise RuntimeError(json.dumps({key: value[:5] for key, value in overlaps.items() if value}, ensure_ascii=False))

def dataframe_sha256(frame, columns):
    payload = frame[columns].astype(str).sort_values(columns).to_csv(index=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def split_count_frame(split_manifest):
    return split_manifest.groupby(["split", "class_name"]).size().unstack(fill_value=0).reindex(["train", "val", "test"])[CLASS_NAMES]

def build_fixed_split():
    df = add_md5(discover_processed_images(DATA_DIR))
    full_counts = df["label"].value_counts().sort_index().reindex(range(NUM_CLASSES), fill_value=0).astype(int).tolist()
    assert len(df) == 1149, len(df)
    assert full_counts == [403, 198, 328, 220], full_counts
    train_frame, tmp_frame = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED, shuffle=True)
    val_frame, test_frame = train_test_split(tmp_frame, test_size=0.50, stratify=tmp_frame["label"], random_state=SEED, shuffle=True)
    train_frame = train_frame.sort_values("rel_path").reset_index(drop=True)
    val_frame = val_frame.sort_values("rel_path").reset_index(drop=True)
    test_frame = test_frame.sort_values("rel_path").reset_index(drop=True)
    train_frame["split"] = "train"
    val_frame["split"] = "val"
    test_frame["split"] = "test"
    assert len(train_frame) == 804, len(train_frame)
    assert len(val_frame) == 172, len(val_frame)
    assert len(test_frame) == 173, len(test_frame)
    validate_source_split(train_frame, val_frame, test_frame)
    manifest = pd.concat([train_frame, val_frame, test_frame], ignore_index=True)
    manifest["split_order"] = manifest["split"].map({"train": 0, "val": 1, "test": 2})
    manifest = manifest.sort_values(["split_order", "label", "rel_path"]).drop(columns=["split_order"]).reset_index(drop=True)
    manifest = manifest[["rel_path", "path", "class_dir", "class_name", "label", "split", "md5"]]
    manifest.to_csv(FIXED_SPLIT_MANIFEST_PATH, index=False)
    return train_frame, val_frame, test_frame, manifest, full_counts

train_df, val_df, test_df, split_manifest, FULL_CLASS_COUNTS = build_fixed_split()
CLASS_COUNTS = train_df["label"].value_counts().sort_index().reindex(range(NUM_CLASSES), fill_value=1).astype(int).tolist()
FIXED_SPLIT_ID = dataframe_sha256(split_manifest, ["rel_path", "label", "split", "md5"])
split_counts = split_count_frame(split_manifest)
print(f"fixed_split_id={FIXED_SPLIT_ID}")
print(f"total={len(split_manifest)} train={len(train_df)} val={len(val_df)} test={len(test_df)}")
display(split_counts)


In [ ]:
LOSS_RUNS = ["baseline_ce", "sce", "ldam", "asl_single_label", "coinfection_margin_asl_old", "gce", "dcs_ce", "dcs_sce", "false_coinfection_cost_ce", "pairwise_coinfection_ranking_ce", "confidence_gated_dcs_ce", "dcs_ldam", "poly_dcs_ce", "attribute_projection_ce"]
LOSS_LABELS = {"baseline_ce": "CE", "sce": "SCE", "ldam": "LDAM", "asl_single_label": "ASLSingleLabel", "coinfection_margin_asl_old": "Old Co-Infection Margin ASL", "gce": "GCE", "dcs_ce": "Directional Co-Infection Suppression CE", "dcs_sce": "Directional Co-Infection Suppression SCE", "false_coinfection_cost_ce": "False-CoInfection Cost CE", "pairwise_coinfection_ranking_ce": "Pairwise Co-Infection Ranking CE", "confidence_gated_dcs_ce": "Confidence-Gated DCS-CE", "dcs_ldam": "DCS-LDAM", "poly_dcs_ce": "Poly-DCS-CE", "attribute_projection_ce": "Attribute-Projection CE"}
LOSS_TYPES = {"baseline_ce": "baseline_existing", "sce": "baseline_existing", "ldam": "baseline_existing", "asl_single_label": "baseline_existing", "coinfection_margin_asl_old": "baseline_existing_custom", "gce": "baseline_existing", "dcs_ce": "new_custom", "dcs_sce": "new_custom", "false_coinfection_cost_ce": "new_custom", "pairwise_coinfection_ranking_ce": "new_custom", "confidence_gated_dcs_ce": "new_custom", "dcs_ldam": "new_custom", "poly_dcs_ce": "new_custom", "attribute_projection_ce": "new_custom"}
LOSS_SOURCES = {"baseline_ce": "PyTorch CrossEntropyLoss", "sce": "Wang et al. ICCV 2019 SCE", "ldam": "Cao et al. NeurIPS 2019 LDAM-DRW", "asl_single_label": "Ridnik et al. ICCV 2021 ASL official ASLSingleLabel", "coinfection_margin_asl_old": "Previous audited co-infection margin ASL variant", "gce": "Zhang and Sabuncu NeurIPS 2018 GCE", "dcs_ce": "Proposed directional co-infection suppression CE", "dcs_sce": "Proposed directional co-infection suppression SCE", "false_coinfection_cost_ce": "Proposed false co-infection expected-cost CE", "pairwise_coinfection_ranking_ce": "Proposed pairwise co-infection ranking CE", "confidence_gated_dcs_ce": "Proposed confidence-gated directional suppression CE", "dcs_ldam": "Proposed LDAM with directional co-infection suppression", "poly_dcs_ce": "Proposed Poly1 CE with directional co-infection suppression", "attribute_projection_ce": "Proposed four-class CE with AMP-safe disease-attribute projection"}

def reduce_loss(loss, reduction):
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(reduction)

class ASLSingleLabel(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, eps=0.1, reduction="mean"):
        super().__init__()
        self.eps = eps
        self.logsoftmax = nn.LogSoftmax(dim=-1)
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.reduction = reduction
    def forward(self, inputs, target):
        target = target.long().view(-1)
        num_classes = inputs.size(-1)
        log_preds = self.logsoftmax(inputs)
        targets = torch.zeros_like(inputs).scatter_(1, target.unsqueeze(1), 1)
        anti_targets = 1 - targets
        xs_pos = torch.exp(log_preds) * targets
        xs_neg = (1 - torch.exp(log_preds)) * anti_targets
        asymmetric_w = torch.pow(1 - xs_pos - xs_neg, self.gamma_pos * targets + self.gamma_neg * anti_targets)
        log_preds = log_preds * asymmetric_w
        if self.eps > 0:
            targets = targets.mul(1 - self.eps).add(self.eps / num_classes)
        loss = -targets.mul(log_preds).sum(dim=-1)
        return reduce_loss(loss, self.reduction)

class CoInfectionMarginASL(nn.Module):
    def __init__(self, margin=0.15, margin_weight=0.30):
        super().__init__()
        self.base = ASLSingleLabel(gamma_pos=0.0, gamma_neg=4.0, eps=0.1, reduction="mean")
        self.margin = margin
        self.margin_weight = margin_weight
    def forward(self, logits, target):
        target = target.long().view(-1)
        base = self.base(logits, target)
        probs = F.softmax(logits, dim=1)
        p_h = probs[:, 0]
        p_bg = probs[:, 1]
        p_wssv = probs[:, 2]
        p_mix = probs[:, 3]
        losses = []
        mask_h = target == 0
        if mask_h.any():
            losses.extend([F.relu(self.margin - p_h[mask_h] + p_bg[mask_h]), F.relu(self.margin - p_h[mask_h] + p_wssv[mask_h]), F.relu(self.margin - p_h[mask_h] + p_mix[mask_h])])
        mask_bg = target == 1
        if mask_bg.any():
            losses.extend([F.relu(self.margin - p_bg[mask_bg] + p_mix[mask_bg]), F.relu(self.margin - p_bg[mask_bg] + p_wssv[mask_bg])])
        mask_wssv = target == 2
        if mask_wssv.any():
            losses.extend([F.relu(self.margin - p_wssv[mask_wssv] + p_mix[mask_wssv]), F.relu(self.margin - p_wssv[mask_wssv] + p_bg[mask_wssv])])
        mask_mix = target == 3
        if mask_mix.any():
            losses.extend([F.relu(self.margin - p_mix[mask_mix] + p_bg[mask_mix]), F.relu(self.margin - p_mix[mask_mix] + p_wssv[mask_mix]), F.relu(self.margin - p_mix[mask_mix] + p_h[mask_mix])])
        margin_loss = torch.cat([loss.reshape(-1) for loss in losses]).mean() if losses else logits.new_tensor(0.0)
        return base + self.margin_weight * margin_loss

class LDAMLoss(nn.Module):
    def __init__(self, cls_num_list, max_m=0.5, s=30.0, reduction="mean"):
        super().__init__()
        m_list = 1.0 / np.sqrt(np.sqrt(np.asarray(cls_num_list, dtype=np.float32)))
        m_list = m_list * (max_m / np.max(m_list))
        self.register_buffer("m_list", torch.tensor(m_list, dtype=torch.float32))
        self.s = s
        self.reduction = reduction
    def forward(self, logits, target):
        target = target.long().view(-1)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, target.unsqueeze(1), True)
        adjusted = logits.clone()
        adjusted[index] = adjusted[index] - self.m_list.to(logits.device)[target]
        output = torch.where(index, adjusted, logits)
        return F.cross_entropy(self.s * output, target, reduction=self.reduction)

class GCELoss(nn.Module):
    def __init__(self, q=0.7, reduction="mean"):
        super().__init__()
        self.q = q
        self.reduction = reduction
    def forward(self, logits, target):
        target = target.long().view(-1)
        p_true = F.softmax(logits, dim=1).gather(1, target.unsqueeze(1)).squeeze(1).clamp_min(1e-8)
        loss = -torch.log(p_true) if abs(self.q) < 1e-8 else (1 - p_true.pow(self.q)) / self.q
        return reduce_loss(loss, self.reduction)

class SCELoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.1, num_classes=NUM_CLASSES, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes
        self.reduction = reduction
    def forward(self, logits, target):
        target = target.long().view(-1)
        ce = F.cross_entropy(logits, target, reduction="none")
        pred = F.softmax(logits, dim=1).clamp(1e-7, 1.0)
        label = F.one_hot(target, self.num_classes).float().to(logits.device).clamp(1e-4, 1.0)
        rce = -torch.sum(pred * torch.log(label), dim=1)
        return reduce_loss(self.alpha * ce + self.beta * rce, self.reduction)

class Poly1CrossEntropyLoss(nn.Module):
    def __init__(self, epsilon=0.5, reduction="mean"):
        super().__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    def forward(self, logits, target):
        target = target.long().view(-1)
        ce = F.cross_entropy(logits, target, reduction="none")
        pt = F.softmax(logits, dim=1).gather(1, target.unsqueeze(1)).squeeze(1)
        return reduce_loss(ce + self.epsilon * (1 - pt), self.reduction)

def directional_penalties(logits, targets, margin_single=0.10, margin_mix=0.05, gated_threshold=None):
    targets = targets.long().view(-1)
    probs = F.softmax(logits, dim=1)
    p_bg = probs[:, 1]
    p_wssv = probs[:, 2]
    p_mix = probs[:, 3]
    single_penalty = torch.zeros_like(targets, dtype=logits.dtype, device=logits.device)
    bg_mask = targets == 1
    if gated_threshold is not None:
        bg_mask = bg_mask & (p_mix > gated_threshold)
    if bg_mask.any():
        single_penalty[bg_mask] = F.relu(margin_single + p_mix[bg_mask] - p_bg[bg_mask]).pow(2)
    wssv_mask = targets == 2
    if gated_threshold is not None:
        wssv_mask = wssv_mask & (p_mix > gated_threshold)
    if wssv_mask.any():
        single_penalty[wssv_mask] = F.relu(margin_single + p_mix[wssv_mask] - p_wssv[wssv_mask]).pow(2)
    mix_penalty = torch.zeros_like(single_penalty)
    mix_mask = targets == 3
    if mix_mask.any():
        strongest_single = torch.maximum(p_bg[mix_mask], p_wssv[mix_mask])
        mix_penalty[mix_mask] = F.relu(margin_mix + strongest_single - p_mix[mix_mask]).pow(2)
    return single_penalty, mix_penalty, probs

class DirectionalCoInfectionSuppressionCE(nn.Module):
    def __init__(self, margin_single=0.10, margin_mix=0.05, lambda_single=0.15, lambda_mix=0.03, lambda_cost=0.05, reduction="mean"):
        super().__init__()
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
        self.lambda_cost = lambda_cost
        self.reduction = reduction
        self.register_buffer("cost", torch.tensor([[0.0, 1.0, 1.0, 1.5], [1.0, 0.0, 1.5, 2.0], [1.0, 1.5, 0.0, 2.5], [1.5, 1.0, 1.0, 0.0]], dtype=torch.float32))
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        single_penalty, mix_penalty, probs = directional_penalties(logits, targets, self.margin_single, self.margin_mix)
        expected_cost = (probs * self.cost.to(logits.device)[targets]).sum(dim=1)
        return reduce_loss(ce + self.lambda_single * single_penalty + self.lambda_mix * mix_penalty + self.lambda_cost * expected_cost, self.reduction)

class DirectionalCoInfectionSuppressionSCE(nn.Module):
    def __init__(self, alpha=1.0, beta=0.1, margin_single=0.10, margin_mix=0.05, lambda_single=0.12, lambda_mix=0.02, reduction="mean"):
        super().__init__()
        self.base = SCELoss(alpha=alpha, beta=beta, num_classes=NUM_CLASSES, reduction="none")
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        single_penalty, mix_penalty, _ = directional_penalties(logits, targets, self.margin_single, self.margin_mix)
        return reduce_loss(base + self.lambda_single * single_penalty + self.lambda_mix * mix_penalty, self.reduction)

class FalseCoInfectionCostCE(nn.Module):
    def __init__(self, lambda_cost=0.08, reduction="mean"):
        super().__init__()
        self.lambda_cost = lambda_cost
        self.reduction = reduction
        self.register_buffer("cost", torch.tensor([[0.0, 1.0, 1.0, 1.5], [1.0, 0.0, 1.2, 2.4], [1.0, 1.2, 0.0, 3.0], [1.2, 1.0, 1.0, 0.0]], dtype=torch.float32))
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        probs = F.softmax(logits, dim=1)
        expected_cost = (probs * self.cost.to(logits.device)[targets]).sum(dim=1)
        return reduce_loss(ce + self.lambda_cost * expected_cost, self.reduction)

class PairwiseCoInfectionRankingCE(nn.Module):
    def __init__(self, margin=0.10, lambda_rank=0.15, reduction="mean"):
        super().__init__()
        self.margin = margin
        self.lambda_rank = lambda_rank
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        single_penalty, _, _ = directional_penalties(logits, targets, self.margin, 0.0)
        return reduce_loss(ce + self.lambda_rank * single_penalty, self.reduction)

class ConfidenceGatedDCSCE(nn.Module):
    def __init__(self, threshold=0.30, margin_single=0.10, margin_mix=0.05, lambda_single=0.20, lambda_mix=0.02, reduction="mean"):
        super().__init__()
        self.threshold = threshold
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        single_penalty, mix_penalty, _ = directional_penalties(logits, targets, self.margin_single, self.margin_mix, gated_threshold=self.threshold)
        return reduce_loss(ce + self.lambda_single * single_penalty + self.lambda_mix * mix_penalty, self.reduction)

class DCSLDAMLoss(nn.Module):
    def __init__(self, cls_num_list, margin_single=0.10, margin_mix=0.05, lambda_single=0.06, lambda_mix=0.01):
        super().__init__()
        self.base = LDAMLoss(cls_num_list, max_m=0.5, s=30.0, reduction="mean")
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        single_penalty, mix_penalty, _ = directional_penalties(logits, targets, self.margin_single, self.margin_mix)
        return base + self.lambda_single * single_penalty.mean() + self.lambda_mix * mix_penalty.mean()

class PolyDCSCE(nn.Module):
    def __init__(self, epsilon=0.5, margin_single=0.10, margin_mix=0.05, lambda_single=0.08, lambda_mix=0.02, reduction="mean"):
        super().__init__()
        self.base = Poly1CrossEntropyLoss(epsilon=epsilon, reduction="none")
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        single_penalty, mix_penalty, _ = directional_penalties(logits, targets, self.margin_single, self.margin_mix)
        return reduce_loss(base + self.lambda_single * single_penalty + self.lambda_mix * mix_penalty, self.reduction)

class AttributeProjectionCE(nn.Module):
    def __init__(self, lambda_attr=0.05, lambda_single=0.10, reduction="mean"):
        super().__init__()
        self.lambda_attr = lambda_attr
        self.lambda_single = lambda_single
        self.reduction = reduction
        self.register_buffer("attrs", torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=torch.float32))
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        probs = F.softmax(logits.float(), dim=1)
        attrs = self.attrs.to(logits.device).float()
        attr_probs = (probs @ attrs).clamp(1e-6, 1 - 1e-6)
        attr_targets = attrs[targets].float()
        attr_loss = -(attr_targets * torch.log(attr_probs) + (1.0 - attr_targets) * torch.log(1.0 - attr_probs)).sum(dim=1).to(ce.dtype)
        p_mix = probs[:, 3].to(ce.dtype)
        single_mask = (targets == 1) | (targets == 2)
        single_penalty = torch.zeros_like(ce)
        if single_mask.any():
            single_penalty[single_mask] = p_mix[single_mask].pow(2)
        return reduce_loss(ce + self.lambda_attr * attr_loss + self.lambda_single * single_penalty, self.reduction)

def make_loss(loss_key, class_counts, num_classes=4):
    if loss_key == "baseline_ce":
        return nn.CrossEntropyLoss()
    if loss_key == "sce":
        return SCELoss(alpha=1.0, beta=0.1, num_classes=num_classes)
    if loss_key == "ldam":
        return LDAMLoss(class_counts, max_m=0.5, s=30.0)
    if loss_key == "asl_single_label":
        return ASLSingleLabel(gamma_pos=0.0, gamma_neg=4.0, eps=0.1, reduction="mean")
    if loss_key == "coinfection_margin_asl_old":
        return CoInfectionMarginASL(margin=0.15, margin_weight=0.30)
    if loss_key == "gce":
        return GCELoss(q=0.7)
    if loss_key == "dcs_ce":
        return DirectionalCoInfectionSuppressionCE()
    if loss_key == "dcs_sce":
        return DirectionalCoInfectionSuppressionSCE()
    if loss_key == "false_coinfection_cost_ce":
        return FalseCoInfectionCostCE()
    if loss_key == "pairwise_coinfection_ranking_ce":
        return PairwiseCoInfectionRankingCE()
    if loss_key == "confidence_gated_dcs_ce":
        return ConfidenceGatedDCSCE(threshold=0.30)
    if loss_key == "dcs_ldam":
        return DCSLDAMLoss(class_counts)
    if loss_key == "poly_dcs_ce":
        return PolyDCSCE(epsilon=0.5)
    if loss_key == "attribute_projection_ce":
        return AttributeProjectionCE()
    raise ValueError(loss_key)

assert set(LOSS_RUNS) == set(LOSS_LABELS) == set(LOSS_TYPES) == set(LOSS_SOURCES)
_synthetic_logits = torch.tensor([[2.5, 0.1, -0.2, -0.5], [0.0, 2.2, 0.3, 0.6], [-0.1, 0.4, 2.0, 0.8], [-0.5, 0.7, 0.8, 2.4]], dtype=torch.float32)
_synthetic_targets = torch.tensor([0, 1, 2, 3], dtype=torch.long)
loss_sanity_rows = []
for loss_key in LOSS_RUNS:
    criterion = make_loss(loss_key, [282, 139, 230, 153], NUM_CLASSES)
    value = criterion(_synthetic_logits, _synthetic_targets)
    assert torch.is_tensor(value), loss_key
    assert value.ndim == 0, loss_key
    assert torch.isfinite(value).item(), loss_key
    loss_sanity_rows.append({"loss_key": loss_key, "loss": LOSS_LABELS[loss_key], "value": float(value.detach().cpu())})
loss_sanity_df = pd.DataFrame(loss_sanity_rows)
loss_sanity_df.to_csv(OUTPUT_DIR / "loss_sanity_check.csv", index=False)
display(loss_sanity_df)


In [ ]:
def metric_dict_from_labels_preds(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "precision": precision_score(y_true, y_pred, average="macro", zero_division=0), "recall": recall_score(y_true, y_pred, average="macro", zero_division=0), "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0), "cohen_kappa": cohen_kappa_score(y_true, y_pred)}

def per_class_metric_dict(y_true, y_pred):
    report = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    out = {}
    for class_name in CLASS_NAMES:
        out[f"{class_name} Precision"] = float(report[class_name]["precision"])
        out[f"{class_name} Recall"] = float(report[class_name]["recall"])
        out[f"{class_name} F1"] = float(report[class_name]["f1-score"])
    return out

def co_infection_error_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    bg_wssv_total = cm[3].sum()
    bg_total = cm[1].sum()
    wssv_total = cm[2].sum()
    return {"BG_WSSV Recall": cm[3, 3] / bg_wssv_total if bg_wssv_total else np.nan, "BG_WSSV to BG": int(cm[3, 1]), "BG_WSSV to WSSV": int(cm[3, 2]), "BG to BG_WSSV": int(cm[1, 3]), "WSSV to BG_WSSV": int(cm[2, 3]), "BG Recall": cm[1, 1] / bg_total if bg_total else np.nan, "WSSV Recall": cm[2, 2] / wssv_total if wssv_total else np.nan}

def all_metric_dict(y_true, y_pred):
    metrics = metric_dict_from_labels_preds(y_true, y_pred)
    metrics.update(per_class_metric_dict(y_true, y_pred))
    metrics.update(co_infection_error_metrics(y_true, y_pred))
    return metrics

def save_confusion_matrix(y_true, y_pred, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

def save_classification_report(y_true, y_pred, out_path):
    report = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(out_path)
    return report

def confusion_records(y_true, y_pred, loss_key, repeat_id, run_seed):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    rows = []
    for true_idx, true_name in enumerate(CLASS_NAMES):
        for pred_idx, pred_name in enumerate(CLASS_NAMES):
            rows.append({"repeat": repeat_id, "seed": run_seed, "model_key": "convnext_tiny_shrimpxnet", "model": "ConvNeXt-Tiny ShrimpXNet baseline", "loss_key": loss_key, "loss": LOSS_LABELS[loss_key], "true_label": true_idx, "true_class": true_name, "pred_label": pred_idx, "pred_class": pred_name, "count": int(cm[true_idx, pred_idx])})
    return rows

def count_params(model):
    return int(sum(param.numel() for param in model.parameters()))

def model_size_mb(path):
    path = Path(path)
    return path.stat().st_size / 1024**2 if path.exists() and path.is_file() else np.nan

def add_metric_fields(result, val_metrics, test_metrics, inf_time, n_test):
    result.update({"Val Accuracy": round(val_metrics["accuracy"], 4), "Val Precision": round(val_metrics["precision"], 4), "Val Recall": round(val_metrics["recall"], 4), "Val F1-Score": round(val_metrics["macro_f1"], 4), "Val Cohen Kappa": round(val_metrics["cohen_kappa"], 4), "Test Accuracy": round(test_metrics["accuracy"], 4), "Accuracy": round(test_metrics["accuracy"], 4), "Test Precision": round(test_metrics["precision"], 4), "Macro Precision": round(test_metrics["precision"], 4), "Test Recall": round(test_metrics["recall"], 4), "Macro Recall": round(test_metrics["recall"], 4), "Test F1-Score": round(test_metrics["macro_f1"], 4), "Macro F1": round(test_metrics["macro_f1"], 4), "Cohen Kappa": round(test_metrics["cohen_kappa"], 4), "Inference Time (s)": round(inf_time, 4), "FPS": round(n_test / inf_time, 2), "Latency (ms)": round((inf_time / n_test) * 1000, 4), "Latency ms per image": round((inf_time / n_test) * 1000, 4)})
    for key, value in test_metrics.items():
        if key.endswith("Precision") or key.endswith("Recall") or key.endswith("F1") or key in ["BG_WSSV Recall", "BG_WSSV to BG", "BG_WSSV to WSSV", "BG to BG_WSSV", "WSSV to BG_WSSV", "BG Recall", "WSSV Recall"]:
            result[key] = round(value, 4) if isinstance(value, float) else value
    return result

def save_prediction_frame(frame, probs, y_true, y_pred, prediction_path, loss_key, repeat_id, run_seed, split_name):
    rows = frame.copy().reset_index(drop=True).rename(columns={"path": "source_path", "md5": "source_md5"})
    rows["repeat"] = repeat_id
    rows["seed"] = run_seed
    rows["model_key"] = "convnext_tiny_shrimpxnet"
    rows["model"] = "ConvNeXt-Tiny ShrimpXNet baseline"
    rows["loss_key"] = loss_key
    rows["loss"] = LOSS_LABELS[loss_key]
    rows["eval_split"] = split_name
    rows["true_label"] = y_true
    rows["pred_label"] = y_pred
    rows["pred_class_name"] = [CLASS_NAMES[int(label)] for label in y_pred]
    rows["correct"] = rows["true_label"].astype(int) == rows["pred_label"].astype(int)
    for idx, class_name in enumerate(CLASS_NAMES):
        rows[f"prob_{class_name}"] = [float(row[idx]) for row in probs]
    rows.to_csv(prediction_path, index=False)
    return rows


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True), transforms.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.90, 1.10), interpolation=InterpolationMode.BICUBIC, antialias=True), transforms.RandomHorizontalFlip(p=0.5), transforms.RandomRotation(degrees=10, interpolation=InterpolationMode.BICUBIC), transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05), transforms.ToTensor(), transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
eval_transform = transforms.Compose([transforms.Resize(236, interpolation=InterpolationMode.BICUBIC, antialias=True), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
CONVNEXT_TRANSFORM_AUDIT = {"train": ["Resize(256,BICUBIC,antialias=True)", "RandomResizedCrop(224,scale=(0.82,1.0),ratio=(0.90,1.10),BICUBIC,antialias=True)", "RandomHorizontalFlip(p=0.5)", "RandomRotation(degrees=10,BICUBIC)", "ColorJitter(brightness=0.10,contrast=0.10,saturation=0.05)", "ToTensor", "Normalize(ImageNet)"], "eval": ["Resize(236,BICUBIC,antialias=True)", "CenterCrop(224)", "ToTensor", "Normalize(ImageNet)"]}
TIMM_ALIASES = {"convnext_tiny": ["convnext_tiny.fb_in22k", "convnext_tiny.fb_in1k", "convnext_tiny"]}

class ShrimpFrameDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"]), int(idx)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

def make_loader(dataset, batch_size, shuffle, generator):
    kwargs = dict(dataset=dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, worker_init_fn=seed_worker if NUM_WORKERS > 0 else None, generator=generator)
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = 4
    return DataLoader(**kwargs)

def make_data_loaders(seed=SEED):
    train_gen = torch.Generator().manual_seed(seed)
    eval_gen = torch.Generator().manual_seed(seed)
    train_loader = make_loader(ShrimpFrameDataset(train_df, train_transform), MICRO_BATCH_SIZE, True, train_gen)
    val_loader = make_loader(ShrimpFrameDataset(val_df, eval_transform), EVAL_BATCH_SIZE, False, eval_gen)
    test_loader = make_loader(ShrimpFrameDataset(test_df, eval_transform), EVAL_BATCH_SIZE, False, eval_gen)
    return train_loader, val_loader, test_loader

def resolve_timm_name(display_name):
    candidates = TIMM_ALIASES.get(display_name, [display_name])
    available = set(timm.list_models(pretrained=False))
    for candidate in candidates:
        if candidate in available:
            return candidate
    pattern_hits = []
    for candidate in candidates:
        pattern_hits.extend(timm.list_models(candidate + "*", pretrained=False))
    if pattern_hits:
        return sorted(pattern_hits)[0]
    raise ValueError(f"No TIMM model found for {display_name}. Tried: {candidates}")

class TimmShrimpXNet(nn.Module):
    def __init__(self, timm_name, pretrained):
        super().__init__()
        try:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        except TypeError:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
            features = self.backbone(sample)
            if isinstance(features, (list, tuple)):
                features = features[-1]
            num_features = features.flatten(1).shape[1]
        self.backbone.train(was_training)
        self.classifier = nn.Sequential(nn.Linear(num_features, 512), nn.ReLU(inplace=True), nn.Dropout(p=0.5), nn.Linear(512, NUM_CLASSES))
    def forward(self, x):
        x = self.backbone(x)
        if isinstance(x, (list, tuple)):
            x = x[-1]
        x = torch.flatten(x, 1)
        return self.classifier(x)

def create_timm_classifier(display_name):
    timm_name = resolve_timm_name(display_name)
    try:
        model = TimmShrimpXNet(timm_name, pretrained=True)
        pretrained = True
    except Exception as pretrained_error:
        print(f"Pretrained weights failed for {display_name} ({timm_name}): {type(pretrained_error).__name__}: {pretrained_error}")
        model = TimmShrimpXNet(timm_name, pretrained=False)
        pretrained = False
    return model.to(device), timm_name, pretrained

def classifier_parameters(model):
    if hasattr(model, "classifier"):
        return list(model.classifier.parameters())
    params = []
    classifier = model.get_classifier() if hasattr(model, "get_classifier") else None
    if isinstance(classifier, nn.Module):
        params = list(classifier.parameters())
    if not params:
        head_tokens = ("classifier", "head", "fc")
        params = [param for name, param in model.named_parameters() if any(token in name.lower() for token in head_tokens)]
    if not params:
        raise RuntimeError("Could not identify classifier/head parameters for warmup.")
    return params

def freeze_backbone_for_warmup(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in classifier_parameters(model):
        param.requires_grad = True

def make_warmup_optimizer(model):
    return optim.Adam([param for param in model.parameters() if param.requires_grad], lr=WARMUP_HEAD_LR)

def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True

def unfreeze_final_backbone_portion(model, display_name):
    backbone = model.backbone if hasattr(model, "backbone") else model
    trainable_modules = []
    for param in model.parameters():
        param.requires_grad = False
    for param in classifier_parameters(model):
        param.requires_grad = True
    if hasattr(backbone, "features") and isinstance(backbone.features, (nn.Sequential, nn.ModuleList, list, tuple)):
        features = backbone.features
        selected = list(features[5:]) if "convnext" in display_name.lower() and len(features) > 5 else list(features[max(0, len(features) - max(1, len(features) // 3)):])
        trainable_modules.extend(selected)
    elif hasattr(backbone, "stages") and isinstance(backbone.stages, (nn.Sequential, nn.ModuleList, list, tuple)):
        stages = backbone.stages
        start = max(0, len(stages) - max(1, len(stages) // 3))
        trainable_modules.extend(list(stages[start:]))
    elif hasattr(backbone, "blocks") and isinstance(backbone.blocks, (nn.Sequential, nn.ModuleList, list, tuple)):
        blocks = backbone.blocks
        start = max(0, len(blocks) - max(1, len(blocks) // 3))
        trainable_modules.extend(list(blocks[start:]))
    else:
        excluded = {"classifier", "head", "fc", "global_pool", "pool", "avgpool"}
        children = [child for name, child in backbone.named_children() if name not in excluded and not name.startswith("head")]
        trainable_modules.extend(children[-2:] if len(children) >= 2 else children)
    for attr in ["norm", "norm_head", "head_norm", "pre_head", "final_conv"]:
        module = getattr(backbone, attr, None)
        if isinstance(module, nn.Module):
            trainable_modules.append(module)
    for module in trainable_modules:
        unfreeze_module(module)
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

def make_finetune_optimizer(model):
    head_param_ids = {id(param) for param in classifier_parameters(model)}
    backbone_params = []
    head_params = []
    for param in model.parameters():
        if not param.requires_grad:
            continue
        if id(param) in head_param_ids:
            head_params.append(param)
        else:
            backbone_params.append(param)
    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": BACKBONE_FINETUNE_LR})
    if head_params:
        param_groups.append({"params": head_params, "lr": HEAD_FINETUNE_LR})
    return optim.Adam(param_groups)

def extract_logits(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (list, tuple)):
        tensors = [item for item in output if isinstance(item, torch.Tensor)]
        if tensors:
            return tensors[-1]
    if hasattr(output, "logits"):
        return output.logits
    raise TypeError(f"Unsupported model output type: {type(output)}")

def forward_with_amp(model, ims):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        return extract_logits(model(ims))

def new_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [ ]:
def predict_pytorch(model, loader, frame, criterion=None, timed=False, prediction_path=None, loss_key=None, repeat_id=None, run_seed=None, split_name=None):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    all_indices = []
    if timed:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        with torch.no_grad():
            for _ in range(10):
                model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        start = time.time()
    else:
        start = None
    with torch.no_grad():
        for ims, gts, idxs in loader:
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = forward_with_amp(model, ims)
            if criterion is not None:
                total_loss += criterion(logits.float(), gts).item() * ims.size(0)
            probs = F.softmax(logits.float(), dim=1)
            preds = torch.argmax(probs, dim=1)
            all_labels.extend(gts.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_indices.extend(idxs.cpu().numpy().tolist())
    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None
    metrics = all_metric_dict(all_labels, all_preds)
    metrics.update({"loss": total_loss / max(1, len(all_labels)) if criterion is not None else None, "elapsed": elapsed, "labels": all_labels, "preds": all_preds})
    prediction_frame = None
    if prediction_path is not None:
        ordered = pd.DataFrame({"dataset_index": all_indices, "true_label": all_labels, "pred_label": all_preds, "probs": all_probs}).sort_values("dataset_index").reset_index(drop=True)
        source_rows = frame.iloc[ordered["dataset_index"].astype(int).tolist()].reset_index(drop=True)
        prediction_frame = save_prediction_frame(source_rows, ordered["probs"].tolist(), ordered["true_label"].astype(int).tolist(), ordered["pred_label"].astype(int).tolist(), prediction_path, loss_key, repeat_id, run_seed, split_name)
    return metrics, prediction_frame

def load_checkpoint(path, model):
    try:
        checkpoint = torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint

def base_result_fields(loss_key, repeat_id, run_seed, timm_name):
    return {"Repeat": repeat_id, "Seed": run_seed, "Repeat Mode": REPEAT_MODE, "Model Key": "convnext_tiny_shrimpxnet", "Model": "ConvNeXt-Tiny ShrimpXNet baseline", "Backend": "timm", "Backend Name": timm_name, "Loss Key": loss_key, "Loss": LOSS_LABELS[loss_key], "Loss Type": LOSS_TYPES[loss_key], "Loss Source": LOSS_SOURCES[loss_key], "Fixed Split ID": FIXED_SPLIT_ID, "Fixed Split Manifest": str(FIXED_SPLIT_MANIFEST_PATH)}

def failed_result(loss_key, repeat_id, run_seed, exc):
    result = base_result_fields(loss_key, repeat_id, run_seed, CONVNEXT_DISPLAY_NAME)
    for column in ["Parameters (M)", "Parameter Count", "Model Size MB", "Training Time (s)", "Best Epoch", "Best Val Loss", "Best Validation Loss", "Best Val Macro F1", "Best validation Macro-F1", "Val Accuracy", "Val Precision", "Val Recall", "Val F1-Score", "Val Cohen Kappa", "Test Accuracy", "Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Cohen Kappa", "BG_WSSV Recall", "BG_WSSV to BG", "BG_WSSV to WSSV", "BG to BG_WSSV", "WSSV to BG_WSSV", "Inference Time (s)", "FPS", "Latency (ms)", "Latency ms per image"]:
        result[column] = np.nan
    result.update({"Selection Metric": "run_failed", "Run Directory": "", "Checkpoint Path": "", "Checkpoint SHA256": "", "Confusion Matrix Path": "", "Classification Report Path": "", "Val Predictions Path": "", "Test Predictions Path": "", "Training History Path": "", "Error": f"{type(exc).__name__}: {exc}"})
    return result

CONVNEXT_MODEL_KEY = "convnext_tiny_shrimpxnet"
CONVNEXT_MODEL_NAME = "ConvNeXt-Tiny ShrimpXNet baseline"
CONVNEXT_DISPLAY_NAME = "convnext_tiny"

def train_convnext_model(loss_key, repeat_id=1):
    run_seed = REPEAT_SEEDS[repeat_id - 1]
    reset_all_seeds(run_seed)
    train_loader, val_loader, test_loader = make_data_loaders(run_seed)
    run_name = f"{CONVNEXT_MODEL_KEY}_{loss_key}_repeat{repeat_id:02d}"
    print("=" * 90)
    print(f"Training {CONVNEXT_MODEL_NAME} | {LOSS_LABELS[loss_key]} | repeat {repeat_id}/{REPEATS} | seed {run_seed}")
    print("=" * 90)
    model, timm_name, pretrained = create_timm_classifier(CONVNEXT_DISPLAY_NAME)
    criterion = make_loss(loss_key, CLASS_COUNTS, NUM_CLASSES).to(device)
    best_path = CHECKPOINTS_DIR / f"best_{run_name}.pth"
    if best_path.exists():
        best_path.unlink()
    best_val_loss = float("inf")
    best_val_f1 = -1.0
    best_epoch = 0
    epochs_no_improve = 0
    history = []
    history_path = REPORT_DIR / f"training_history_{run_name}.csv"
    train_start = time.time()
    freeze_backbone_for_warmup(model)
    optimizer = make_warmup_optimizer(model)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
    scaler = new_grad_scaler()
    for epoch in range(EPOCHS):
        if epoch == WARMUP_EPOCHS:
            trainable_count = unfreeze_final_backbone_portion(model, CONVNEXT_DISPLAY_NAME)
            print(f"Fine-tune trainable parameters: {trainable_count:,}")
            optimizer = make_finetune_optimizer(model)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
            epochs_no_improve = 0
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        current_lrs = [group["lr"] for group in optimizer.param_groups]
        optimizer.zero_grad(set_to_none=True)
        for step, (ims, gts, _) in enumerate(tqdm(train_loader, desc=f"{CONVNEXT_DISPLAY_NAME} {LOSS_LABELS[loss_key]} repeat {repeat_id} epoch {epoch + 1}/{EPOCHS}", leave=False)):
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = extract_logits(model(ims))
                loss = criterion(logits.float(), gts)
                scaled_loss = loss / ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            train_loss += loss.item() * ims.size(0)
            train_correct += (torch.argmax(logits, dim=1) == gts).sum().item()
            train_total += gts.size(0)
        scheduler.step()
        val_metrics, _ = predict_pytorch(model, val_loader, val_df, criterion=criterion, timed=False)
        train_acc = train_correct / max(1, train_total)
        phase = "warmup" if epoch < WARMUP_EPOCHS else "finetune"
        lr_text = ",".join(f"{lr:.2e}" for lr in current_lrs)
        epoch_record = {"Repeat": repeat_id, "Seed": run_seed, "Model Key": CONVNEXT_MODEL_KEY, "Model": CONVNEXT_MODEL_NAME, "Backend": "timm", "Loss": LOSS_LABELS[loss_key], "Loss Key": loss_key, "Epoch": epoch + 1, "Phase": phase, "LR": lr_text, "Train Loss": train_loss / max(1, train_total), "Train Accuracy": train_acc, "Val Loss": val_metrics["loss"], "Val Accuracy": val_metrics["accuracy"], "Val Precision": val_metrics["precision"], "Val Recall": val_metrics["recall"], "Val F1-Score": val_metrics["macro_f1"], "Val Cohen Kappa": val_metrics["cohen_kappa"]}
        history.append(epoch_record)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Phase: {phase} | LR: {lr_text} | Train Loss: {epoch_record['Train Loss']:.4f} - Acc: {train_acc:.4f} | Val Loss: {val_metrics['loss']:.4f} - Acc: {val_metrics['accuracy']:.4f} - Macro F1: {val_metrics['macro_f1']:.4f}")
        improved = (val_metrics["macro_f1"] > best_val_f1) or (val_metrics["macro_f1"] == best_val_f1 and val_metrics["loss"] < best_val_loss)
        if improved:
            best_val_loss = val_metrics["loss"]
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch + 1
            epochs_no_improve = 0
            torch.save({"model_state_dict": model.state_dict(), "display_name": CONVNEXT_DISPLAY_NAME, "model_key": CONVNEXT_MODEL_KEY, "timm_name": timm_name, "backend_source": "timm", "pretrained": pretrained, "loss_key": loss_key, "loss_label": LOSS_LABELS[loss_key], "num_classes": NUM_CLASSES, "img_size": IMG_SIZE, "class_names": CLASS_NAMES, "class_dirs": CLASS_DIRS, "seed": run_seed, "repeat": repeat_id, "split_manifest": str(FIXED_SPLIT_MANIFEST_PATH), "fixed_split_id": FIXED_SPLIT_ID, "transforms": CONVNEXT_TRANSFORM_AUDIT}, best_path)
            print(f"Saved best checkpoint: val macro F1 {best_val_f1:.4f}, val loss {best_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break
    train_time = time.time() - train_start
    checkpoint = load_checkpoint(best_path, model)
    val_prediction_path = REPORT_DIR / f"val_predictions_{run_name}.csv"
    test_prediction_path = REPORT_DIR / f"test_predictions_{run_name}.csv"
    val_metrics, val_predictions = predict_pytorch(model, val_loader, val_df, criterion=criterion, timed=False, prediction_path=val_prediction_path, loss_key=loss_key, repeat_id=repeat_id, run_seed=run_seed, split_name="val")
    test_metrics, test_predictions = predict_pytorch(model, test_loader, test_df, criterion=None, timed=True, prediction_path=test_prediction_path, loss_key=loss_key, repeat_id=repeat_id, run_seed=run_seed, split_name="test")
    inf_time = max(test_metrics["elapsed"], 1e-9)
    param_count = count_params(model)
    cm_path = FIGURES_DIR / f"confusion_matrix_{run_name}.png"
    report_path = REPORT_DIR / f"classification_report_{run_name}.csv"
    save_confusion_matrix(test_metrics["labels"], test_metrics["preds"], f"{CONVNEXT_MODEL_NAME} | {LOSS_LABELS[loss_key]} | repeat {repeat_id}", cm_path)
    save_classification_report(test_metrics["labels"], test_metrics["preds"], report_path)
    confusion_rows = confusion_records(test_metrics["labels"], test_metrics["preds"], loss_key, repeat_id, run_seed)
    result = base_result_fields(loss_key, repeat_id, run_seed, timm_name)
    result.update({"Selection Metric": "val_macro_f1_then_val_loss", "Parameters (M)": round(param_count / 1e6, 2), "Parameter Count": param_count, "Model Size MB": round(model_size_mb(best_path), 3), "Training Time (s)": round(train_time, 1), "Best Epoch": best_epoch, "Best Val Loss": round(best_val_loss, 6), "Best Validation Loss": round(best_val_loss, 6), "Best Val Macro F1": round(best_val_f1, 6), "Best validation Macro-F1": round(best_val_f1, 6), "Run Directory": str(CHECKPOINTS_DIR), "Checkpoint Path": str(best_path), "Checkpoint SHA256": sha256_file(best_path), "Confusion Matrix Path": str(cm_path), "Classification Report Path": str(report_path), "Val Predictions Path": str(val_prediction_path), "Test Predictions Path": str(test_prediction_path), "Training History Path": str(history_path), "Pretrained": pretrained, "Error": ""})
    result = add_metric_fields(result, val_metrics, test_metrics, inf_time, len(test_df))
    del model, optimizer, scheduler, criterion, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result, history, confusion_rows, [val_predictions, test_predictions]


In [ ]:
def environment_versions():
    payload = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "python_version": sys.version, "python_executable": sys.executable, "platform": platform.platform(), "torch_version": torch.__version__, "torch_cuda_version": str(torch.version.cuda), "torch_cudnn_version": str(torch.backends.cudnn.version()), "torchvision_available": importlib.util.find_spec("torchvision") is not None, "cuda_available": torch.cuda.is_available(), "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "", "timm_version": timm.__version__, "sklearn_version": sklearn.__version__, "numpy_version": np.__version__, "pandas_version": pd.__version__, "pillow_version": Image.__version__, "nvidia_smi": nvidia_smi_text()}
    save_json(ENVIRONMENT_VERSIONS_PATH, payload)
    return payload

def build_run_audit(stage):
    payload = {"stage": stage, "timestamp_utc": datetime.now(timezone.utc).isoformat(), "data_dir": str(DATA_DIR), "audited_yolo_expected_data_dir": str(DEFAULT_DATA_DIR), "output_dir": str(OUTPUT_DIR), "reports_dir": str(REPORT_DIR), "figures_dir": str(FIGURES_DIR), "checkpoints_dir": str(CHECKPOINTS_DIR), "class_dirs": CLASS_DIRS, "class_names": CLASS_NAMES, "class_to_idx": CLASS_TO_IDX, "num_classes": NUM_CLASSES, "full_class_counts": FULL_CLASS_COUNTS, "train_class_counts": CLASS_COUNTS, "split": {"method": "train_test_split", "random_state": SEED, "stratify": "label", "train": len(train_df), "val": len(val_df), "test": len(test_df), "fixed_split_id": FIXED_SPLIT_ID, "manifest": str(FIXED_SPLIT_MANIFEST_PATH)}, "repeat_count": REPEATS, "repeat_seeds": REPEAT_SEEDS, "loss_runs": LOSS_RUNS, "loss_labels": LOSS_LABELS, "loss_types": LOSS_TYPES, "shrimpxnet_ipynb_status": "not_found_in_repository_at_generation_time", "reference_notebooks_inspected": ["legacy/final_yolo26m_convnext_tiny_CE_ASL_5asl_losses_best_pipeline_rtx4090.ipynb", "best/final_yolo26m_convnext_tiny_CE_ASL_original_yolo_best_repeat3.ipynb"], "convnext": {"model_key": CONVNEXT_MODEL_KEY, "display_name": CONVNEXT_MODEL_NAME, "model_source": "timm", "model_aliases": TIMM_ALIASES[CONVNEXT_DISPLAY_NAME], "pretrained_weight_approach": "timm pretrained=True with fallback to pretrained=False", "img_size": IMG_SIZE, "paper_batch_size": PAPER_BATCH_SIZE, "micro_batch_size": MICRO_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS, "eval_batch_size": EVAL_BATCH_SIZE, "optimizer": "Adam", "warmup_epochs": WARMUP_EPOCHS, "warmup_head_lr": WARMUP_HEAD_LR, "backbone_finetune_lr": BACKBONE_FINETUNE_LR, "head_finetune_lr": HEAD_FINETUNE_LR, "scheduler": "StepLR", "step_size": STEP_SIZE, "step_gamma": STEP_GAMMA, "patience": PATIENCE, "checkpoint_selection": "validation macro F1 then validation loss", "transforms": CONVNEXT_TRANSFORM_AUDIT, "imagenet_mean": IMAGENET_MEAN, "imagenet_std": IMAGENET_STD}, "runtime_notes": {"target": "Linux RTX 4090 24GB", "full_training_runs": len(LOSS_RUNS), "attribute_projection_ce": "manual float32 BCE term avoids torch.nn.functional.binary_cross_entropy under AMP", "partial_state_collection": "empty paths and directories are skipped"}}
    save_json(RUN_AUDIT_PATH, payload)
    return payload

def collect_existing_predictions(summary_frame):
    frames = []
    if summary_frame is None or summary_frame.empty:
        return pd.DataFrame()
    for column in ["Val Predictions Path", "Test Predictions Path"]:
        if column not in summary_frame.columns:
            continue
        for value in summary_frame[column].dropna().astype(str).tolist():
            if not value.strip():
                continue
            p = Path(value)
            if p.is_file() and p.stat().st_size > 0:
                frames.append(pd.read_csv(p))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def save_partial_state(results, histories, confusions):
    summary_frame = pd.DataFrame(results)
    history_frame = pd.DataFrame(histories)
    confusion_frame = pd.DataFrame(confusions)
    predictions_frame = collect_existing_predictions(summary_frame)
    summary_frame.to_csv(LOSS_RUN_SUMMARY_RAW_PATH, index=False)
    summary_frame.to_json(OUTPUT_DIR / "loss_run_summary_raw.json", orient="records", indent=2)
    history_frame.to_csv(OUTPUT_DIR / "training_history_all_runs.csv", index=False)
    confusion_frame.to_csv(OUTPUT_DIR / "confusion_matrix_per_run.csv", index=False)
    predictions_frame.to_csv(ALL_PREDICTIONS_PATH, index=False)
    return summary_frame, history_frame, confusion_frame, predictions_frame

env_payload = environment_versions()
initial_audit = build_run_audit("before_training")
RUN_SPECS = [{"repeat_id": repeat_id, "loss_key": loss_key} for repeat_id in range(1, REPEATS + 1) for loss_key in LOSS_RUNS]
pd.DataFrame(RUN_SPECS).to_csv(OUTPUT_DIR / "run_plan.csv", index=False)
comparison_results = []
all_histories = []
all_confusions = []
if RUN_TRAINING:
    for spec in RUN_SPECS:
        repeat_id = spec["repeat_id"]
        loss_key = spec["loss_key"]
        run_seed = REPEAT_SEEDS[repeat_id - 1]
        try:
            result, history, confusions, _ = train_convnext_model(loss_key, repeat_id)
            comparison_results.append(result)
            all_histories.extend(history)
            all_confusions.extend(confusions)
            display(pd.DataFrame([result]))
        except Exception as exc:
            result = failed_result(loss_key, repeat_id, run_seed, exc)
            comparison_results.append(result)
            print(f"ERROR convnext_tiny_shrimpxnet {loss_key} repeat {repeat_id}: {type(exc).__name__}: {exc}")
            display(pd.DataFrame([result]))
        save_partial_state(comparison_results, all_histories, all_confusions)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
else:
    print("RUN_TRAINING=0, training loop skipped")
df_summary, history_df, confusion_df, all_predictions = save_partial_state(comparison_results, all_histories, all_confusions)


In [ ]:
def ensure_summary_columns(frame):
    required = ["Repeat", "Seed", "Model Key", "Model", "Backend", "Loss Key", "Loss", "Loss Type", "Selection Metric", "Parameters (M)", "Parameter Count", "Model Size MB", "Training Time (s)", "Best Epoch", "Best Val Loss", "Best Validation Loss", "Best Val Macro F1", "Best validation Macro-F1", "Val Accuracy", "Val Precision", "Val Recall", "Val F1-Score", "Val Cohen Kappa", "Test Accuracy", "Accuracy", "Test Precision", "Macro Precision", "Test Recall", "Macro Recall", "Test F1-Score", "Macro F1", "Cohen Kappa", "BG_WSSV Recall", "BG_WSSV to BG", "BG_WSSV to WSSV", "BG to BG_WSSV", "WSSV to BG_WSSV", "Inference Time (s)", "FPS", "Latency (ms)", "Latency ms per image", "Checkpoint Path", "Confusion Matrix Path", "Classification Report Path", "Val Predictions Path", "Test Predictions Path", "Training History Path", "Error"]
    out = frame.copy()
    for column in required:
        if column not in out.columns:
            out[column] = np.nan
    return out

def aggregate_group_stats(frame):
    numeric_metrics = ["Best Epoch", "Best Val Loss", "Best Val Macro F1", "Val Accuracy", "Val Precision", "Val Recall", "Val F1-Score", "Val Cohen Kappa", "Test Accuracy", "Test Precision", "Test Recall", "Test F1-Score", "Cohen Kappa", "BG_WSSV Recall", "BG_WSSV to BG", "BG_WSSV to WSSV", "BG to BG_WSSV", "WSSV to BG_WSSV", "BG Recall", "WSSV Recall", "Inference Time (s)", "FPS", "Latency (ms)", "Model Size MB", "Parameters (M)", "Parameter Count"]
    rows = []
    if frame.empty:
        return pd.DataFrame()
    for (loss_key, loss), group in frame.groupby(["Loss Key", "Loss"], dropna=False):
        row = {"Model Key": CONVNEXT_MODEL_KEY, "Model": CONVNEXT_MODEL_NAME, "Backend": "timm", "Loss Key": loss_key, "Loss": loss, "Loss Type": LOSS_TYPES.get(loss_key, ""), "Runs": len(group), "Completed Runs": int(pd.to_numeric(group["Test F1-Score"], errors="coerce").notna().sum()), "Repeat Mode": REPEAT_MODE}
        for metric in numeric_metrics:
            values = pd.to_numeric(group[metric], errors="coerce").dropna() if metric in group.columns else pd.Series(dtype=float)
            row[f"{metric} Mean"] = values.mean() if len(values) else np.nan
            row[f"{metric} Std"] = values.std(ddof=1) if len(values) > 1 else 0.0 if len(values) == 1 else np.nan
            row[f"{metric} Min"] = values.min() if len(values) else np.nan
            row[f"{metric} Max"] = values.max() if len(values) else np.nan
        rows.append(row)
    out = pd.DataFrame(rows)
    sort_columns = ["Test F1-Score Mean", "Cohen Kappa Mean", "BG_WSSV Recall Mean", "WSSV to BG_WSSV Mean", "BG to BG_WSSV Mean"]
    for column in sort_columns:
        if column not in out.columns:
            out[column] = np.nan
    out = out.sort_values(by=sort_columns, ascending=[False, False, False, True, True], na_position="last").reset_index(drop=True)
    out["Rank"] = np.arange(1, len(out) + 1)
    return out

def deltas_vs_ce(group_stats):
    if group_stats.empty:
        return pd.DataFrame()
    ce = group_stats[group_stats["Loss Key"] == "baseline_ce"]
    if ce.empty:
        return pd.DataFrame()
    ce_row = ce.iloc[0]
    rows = []
    for _, row in group_stats.iterrows():
        rows.append({"Loss Key": row["Loss Key"], "Loss": row["Loss"], "Test F1 Mean": row.get("Test F1-Score Mean", np.nan), "Cohen Kappa Mean": row.get("Cohen Kappa Mean", np.nan), "Test Accuracy Mean": row.get("Test Accuracy Mean", np.nan), "BG_WSSV Recall Mean": row.get("BG_WSSV Recall Mean", np.nan), "WSSV to BG_WSSV Mean": row.get("WSSV to BG_WSSV Mean", np.nan), "BG to BG_WSSV Mean": row.get("BG to BG_WSSV Mean", np.nan), "Delta F1 vs CE": row.get("Test F1-Score Mean", np.nan) - ce_row.get("Test F1-Score Mean", np.nan), "Delta Kappa vs CE": row.get("Cohen Kappa Mean", np.nan) - ce_row.get("Cohen Kappa Mean", np.nan), "Delta Accuracy vs CE": row.get("Test Accuracy Mean", np.nan) - ce_row.get("Test Accuracy Mean", np.nan), "Delta WSSV to BG_WSSV vs CE": row.get("WSSV to BG_WSSV Mean", np.nan) - ce_row.get("WSSV to BG_WSSV Mean", np.nan), "Delta BG to BG_WSSV vs CE": row.get("BG to BG_WSSV Mean", np.nan) - ce_row.get("BG to BG_WSSV Mean", np.nan)})
    return pd.DataFrame(rows)

def choose_best(group_stats, mask=None):
    frame = group_stats.copy()
    if mask is not None:
        frame = frame[mask(frame)].copy()
    if frame.empty or "Test F1-Score Mean" not in frame.columns:
        return {}
    frame = frame[pd.to_numeric(frame["Test F1-Score Mean"], errors="coerce").notna()].copy()
    if frame.empty:
        return {}
    frame = frame.sort_values(by=["Test F1-Score Mean", "Cohen Kappa Mean", "BG_WSSV Recall Mean", "WSSV to BG_WSSV Mean", "BG to BG_WSSV Mean"], ascending=[False, False, False, True, True])
    return frame.iloc[0].to_dict()

def build_output_table_audit(paths):
    rows = []
    for label, path in paths:
        p = Path(path)
        row = {"artifact": label, "path": str(p), "exists": p.exists(), "is_file": p.is_file(), "size_bytes": p.stat().st_size if p.exists() and p.is_file() else 0, "sha256": sha256_file(p) if p.exists() and p.is_file() else ""}
        if p.exists() and p.is_file() and p.suffix.lower() == ".csv":
            try:
                row["rows"] = len(pd.read_csv(p))
            except Exception:
                row["rows"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def zip_figures_and_reports():
    with zipfile.ZipFile(FIGURES_AND_REPORTS_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for base in [REPORT_DIR, FIGURES_DIR]:
            if base.exists():
                for path in sorted(base.rglob("*")):
                    if path.is_file():
                        zf.write(path, path.relative_to(OUTPUT_DIR))
        for path in [LOSS_RUN_SUMMARY_RAW_PATH, LOSS_GROUP_STATS_PATH, LOSS_DELTAS_VS_CE_PATH, BG_WSSV_ERROR_SUMMARY_PATH, ALL_PREDICTIONS_PATH, FIXED_SPLIT_MANIFEST_PATH, RUN_AUDIT_PATH, ENVIRONMENT_VERSIONS_PATH]:
            if path.exists() and path.is_file():
                zf.write(path, path.relative_to(OUTPUT_DIR))

if "df_summary" not in globals() or df_summary.empty:
    df_summary = pd.read_csv(LOSS_RUN_SUMMARY_RAW_PATH) if LOSS_RUN_SUMMARY_RAW_PATH.exists() else pd.DataFrame()
if "history_df" not in globals() or history_df.empty:
    history_df = pd.read_csv(OUTPUT_DIR / "training_history_all_runs.csv") if (OUTPUT_DIR / "training_history_all_runs.csv").exists() else pd.DataFrame()
if "confusion_df" not in globals() or confusion_df.empty:
    confusion_df = pd.read_csv(OUTPUT_DIR / "confusion_matrix_per_run.csv") if (OUTPUT_DIR / "confusion_matrix_per_run.csv").exists() else pd.DataFrame()
df_summary = ensure_summary_columns(df_summary)
df_summary.to_csv(LOSS_RUN_SUMMARY_RAW_PATH, index=False)
group_stats = aggregate_group_stats(df_summary)
loss_deltas_vs_ce = deltas_vs_ce(group_stats)
bg_wssv_error_columns = ["Rank", "Model Key", "Model", "Loss Key", "Loss", "Loss Type", "Runs", "Completed Runs", "BG_WSSV Recall Mean", "BG_WSSV to BG Mean", "BG_WSSV to WSSV Mean", "BG to BG_WSSV Mean", "WSSV to BG_WSSV Mean", "BG Recall Mean", "WSSV Recall Mean"]
bg_wssv_error_summary = group_stats[[column for column in bg_wssv_error_columns if column in group_stats.columns]].copy() if not group_stats.empty else pd.DataFrame(columns=bg_wssv_error_columns)
all_predictions = collect_existing_predictions(df_summary)
group_stats.to_csv(LOSS_GROUP_STATS_PATH, index=False)
loss_deltas_vs_ce.to_csv(LOSS_DELTAS_VS_CE_PATH, index=False)
bg_wssv_error_summary.to_csv(BG_WSSV_ERROR_SUMMARY_PATH, index=False)
all_predictions.to_csv(ALL_PREDICTIONS_PATH, index=False)
final_summary_payload = {"best_overall": choose_best(group_stats), "best_custom_loss": choose_best(group_stats, lambda frame: frame["Loss Type"].isin(["new_custom", "baseline_existing_custom"])), "best_robust_noisy_label_loss": choose_best(group_stats, lambda frame: frame["Loss Key"].isin(["sce", "gce", "asl_single_label", "ldam"])), "best_coinfection_suppression_loss": choose_best(group_stats, lambda frame: frame["Loss Key"].isin(["coinfection_margin_asl_old", "dcs_ce", "dcs_sce", "false_coinfection_cost_ce", "pairwise_coinfection_ranking_ce", "confidence_gated_dcs_ce", "dcs_ldam", "poly_dcs_ce", "attribute_projection_ce"])), "ranking_rule": ["Test Macro-F1", "Cohen Kappa", "BG_WSSV Recall", "Fewer WSSV to BG_WSSV errors", "Fewer BG to BG_WSSV errors"], "fixed_split_id": FIXED_SPLIT_ID, "run_count": int(len(df_summary)), "completed_run_count": int(pd.to_numeric(df_summary["Test F1-Score"], errors="coerce").notna().sum()) if not df_summary.empty else 0}
save_json(FINAL_SUMMARY_JSON_PATH, final_summary_payload)
final_audit = build_run_audit("after_training")
zip_figures_and_reports()
required_artifacts = [("run_audit", RUN_AUDIT_PATH), ("environment_versions", ENVIRONMENT_VERSIONS_PATH), ("fixed_split_manifest", FIXED_SPLIT_MANIFEST_PATH), ("loss_run_summary_raw", LOSS_RUN_SUMMARY_RAW_PATH), ("loss_group_stats", LOSS_GROUP_STATS_PATH), ("loss_deltas_vs_ce", LOSS_DELTAS_VS_CE_PATH), ("bg_wssv_error_summary", BG_WSSV_ERROR_SUMMARY_PATH), ("all_predictions", ALL_PREDICTIONS_PATH), ("final_summary_json", FINAL_SUMMARY_JSON_PATH), ("figures_and_reports_zip", FIGURES_AND_REPORTS_ZIP_PATH)]
for path in sorted(REPORT_DIR.glob("classification_report_*.csv")):
    required_artifacts.append((f"report_{path.stem}", path))
for path in sorted(REPORT_DIR.glob("test_predictions_*.csv")):
    required_artifacts.append((f"predictions_{path.stem}", path))
for path in sorted(REPORT_DIR.glob("val_predictions_*.csv")):
    required_artifacts.append((f"predictions_{path.stem}", path))
for path in sorted(REPORT_DIR.glob("training_history_*.csv")):
    required_artifacts.append((f"history_{path.stem}", path))
for path in sorted(FIGURES_DIR.glob("confusion_matrix_*.png")):
    required_artifacts.append((f"figure_{path.stem}", path))
output_table_audit = build_output_table_audit(required_artifacts)
output_table_audit.to_csv(OUTPUT_TABLE_AUDIT_PATH, index=False)
run_config_df = pd.DataFrame([{"seed": SEED, "repeats": REPEATS, "img_size": IMG_SIZE, "epochs": EPOCHS, "patience": PATIENCE, "convnext_micro_batch_size": MICRO_BATCH_SIZE, "convnext_effective_batch_size": MICRO_BATCH_SIZE * ACCUMULATION_STEPS, "convnext_accumulation_steps": ACCUMULATION_STEPS, "fixed_split_id": FIXED_SPLIT_ID}])
environment_df = pd.DataFrame([env_payload])
loss_config_df = pd.DataFrame([{"loss_key": key, "loss": LOSS_LABELS[key], "loss_type": LOSS_TYPES[key], "source": LOSS_SOURCES[key]} for key in LOSS_RUNS])
with pd.ExcelWriter(FINAL_SUMMARY_XLSX_PATH, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="raw_runs", index=False)
    group_stats.to_excel(writer, sheet_name="group_stats", index=False)
    loss_deltas_vs_ce.to_excel(writer, sheet_name="deltas", index=False)
    bg_wssv_error_summary.to_excel(writer, sheet_name="bg_wssv_errors", index=False)
    output_table_audit.to_excel(writer, sheet_name="output_table_audit", index=False)
    split_counts.reset_index().to_excel(writer, sheet_name="split_counts", index=False)
    environment_df.to_excel(writer, sheet_name="environment", index=False)
    run_config_df.to_excel(writer, sheet_name="run_config", index=False)
    loss_config_df.to_excel(writer, sheet_name="loss_config", index=False)
    history_df.to_excel(writer, sheet_name="history", index=False)
    confusion_df.to_excel(writer, sheet_name="confusions", index=False)
required_artifacts.append(("final_summary_xlsx", FINAL_SUMMARY_XLSX_PATH))
required_artifacts.append(("output_table_audit", OUTPUT_TABLE_AUDIT_PATH))
output_table_audit = build_output_table_audit(required_artifacts)
output_table_audit.to_csv(OUTPUT_TABLE_AUDIT_PATH, index=False)
display(group_stats)
print(FINAL_SUMMARY_XLSX_PATH)
